<a href="https://colab.research.google.com/github/Barna036/Intro-to-Python1/blob/main/Day_4_API_Practice_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 API practice

Run these from top to bottom. Sections are separated by markdown headers so you can collapse them. FRED requires an API key — see that section for setup.

## Setup

In [29]:
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "ICPSR-Python-Course/1.0 (willh212@gmail.com)"
}

## Wikipedia: Jimmy Carter, language link count

In [28]:
endpoint = "https://en.wikipedia.org/w/api.php"

parameters = {
    "action":  "query",
    "titles":  "Jimmy_Carter",
    "prop":    "langlinkscount",
    "format":  "json",
}

r = requests.get(endpoint, params=parameters, headers=HEADERS)
print("Status:", r.status_code)
r.json()

Status: 200


{'batchcomplete': '',
 'query': {'normalized': [{'from': 'Jimmy_Carter', 'to': 'Jimmy Carter'}],
  'pages': {'15992': {'pageid': 15992,
    'ns': 0,
    'title': 'Jimmy Carter',
    'langlinkscount': 164}}}}

### Unpacking the response

The API returned a nested dictionary. Before we extract what we want, let's see how it's structured

In [27]:
d = r.json()

# Level 0: the top-level keys
d.keys()

dict_keys(['batchcomplete', 'query'])

`batchcomplete` is bookkeeping — Wikipedia's way of saying "your whole query finished." You can ignore it.

Everything we care about lives under `query`.

In [26]:
# Level 1: inside query
d["query"].keys()

dict_keys(['normalized', 'pages'])

`normalized` is Wikipedia telling us it treated `Jimmy_Carter` (with an underscore) as `Jimmy Carter` (with a space). Useful for debugging, but not what we want.

`pages` holds the actual data.

In [14]:
# Level 2: pages
d["query"]["pages"]

{'15992': {'pageid': 15992,
  'ns': 0,
  'title': 'Jimmy Carter',
  'pageviews': {'2026-05-11': 7251,
   '2026-05-12': 7336,
   '2026-05-13': 7281,
   '2026-05-14': 7219,
   '2026-05-15': 6496,
   '2026-05-16': 6009,
   '2026-05-17': 6740,
   '2026-05-18': 6895,
   '2026-05-19': 6708,
   '2026-05-20': 6781,
   '2026-05-21': 6493,
   '2026-05-22': 6087,
   '2026-05-23': 5881,
   '2026-05-24': 6231,
   '2026-05-25': 6465,
   '2026-05-26': 6668,
   '2026-05-27': 6516,
   '2026-05-28': 6673,
   '2026-05-29': 7726,
   '2026-05-30': 6780,
   '2026-05-31': 6481,
   '2026-06-01': 6355,
   '2026-06-02': 6387,
   '2026-06-03': 6414,
   '2026-06-04': 6074,
   '2026-06-05': 6415,
   '2026-06-06': 5758,
   '2026-06-07': 6023,
   '2026-06-08': 5795,
   '2026-06-09': 5695,
   '2026-06-10': 5789,
   '2026-06-11': 5606,
   '2026-06-12': 5746,
   '2026-06-13': 5750,
   '2026-06-14': 6446,
   '2026-06-15': 6455,
   '2026-06-16': 5958,
   '2026-06-17': 5853,
   '2026-06-18': 6829,
   '2026-06-19': 8411,
 

**Here's the awkward bit:** the key inside `pages` is Wikipedia's internal **page ID** (`"15992"` for Jimmy Carter). Page IDs are permanent, but you don't know them in advance.

We want the *value* (the dict with `pageid`, `title`, `langlinkscount`) without hardcoding the ID.

In [13]:
# Grab just the values and throw away the page-ID keys
list(d["query"]["pages"].values())

[{'pageid': 15992,
  'ns': 0,
  'title': 'Jimmy Carter',
  'pageviews': {'2026-05-11': 7251,
   '2026-05-12': 7336,
   '2026-05-13': 7281,
   '2026-05-14': 7219,
   '2026-05-15': 6496,
   '2026-05-16': 6009,
   '2026-05-17': 6740,
   '2026-05-18': 6895,
   '2026-05-19': 6708,
   '2026-05-20': 6781,
   '2026-05-21': 6493,
   '2026-05-22': 6087,
   '2026-05-23': 5881,
   '2026-05-24': 6231,
   '2026-05-25': 6465,
   '2026-05-26': 6668,
   '2026-05-27': 6516,
   '2026-05-28': 6673,
   '2026-05-29': 7726,
   '2026-05-30': 6780,
   '2026-05-31': 6481,
   '2026-06-01': 6355,
   '2026-06-02': 6387,
   '2026-06-03': 6414,
   '2026-06-04': 6074,
   '2026-06-05': 6415,
   '2026-06-06': 5758,
   '2026-06-07': 6023,
   '2026-06-08': 5795,
   '2026-06-09': 5695,
   '2026-06-10': 5789,
   '2026-06-11': 5606,
   '2026-06-12': 5746,
   '2026-06-13': 5750,
   '2026-06-14': 6446,
   '2026-06-15': 6455,
   '2026-06-16': 5958,
   '2026-06-17': 5853,
   '2026-06-18': 6829,
   '2026-06-19': 8411,
   '2026-0

That's a list containing one dictionary. Grab the first (and only) item:

In [12]:
# The most intuitive way
list(d["query"]["pages"].values())[0]

{'pageid': 15992,
 'ns': 0,
 'title': 'Jimmy Carter',
 'pageviews': {'2026-05-11': 7251,
  '2026-05-12': 7336,
  '2026-05-13': 7281,
  '2026-05-14': 7219,
  '2026-05-15': 6496,
  '2026-05-16': 6009,
  '2026-05-17': 6740,
  '2026-05-18': 6895,
  '2026-05-19': 6708,
  '2026-05-20': 6781,
  '2026-05-21': 6493,
  '2026-05-22': 6087,
  '2026-05-23': 5881,
  '2026-05-24': 6231,
  '2026-05-25': 6465,
  '2026-05-26': 6668,
  '2026-05-27': 6516,
  '2026-05-28': 6673,
  '2026-05-29': 7726,
  '2026-05-30': 6780,
  '2026-05-31': 6481,
  '2026-06-01': 6355,
  '2026-06-02': 6387,
  '2026-06-03': 6414,
  '2026-06-04': 6074,
  '2026-06-05': 6415,
  '2026-06-06': 5758,
  '2026-06-07': 6023,
  '2026-06-08': 5795,
  '2026-06-09': 5695,
  '2026-06-10': 5789,
  '2026-06-11': 5606,
  '2026-06-12': 5746,
  '2026-06-13': 5750,
  '2026-06-14': 6446,
  '2026-06-15': 6455,
  '2026-06-16': 5958,
  '2026-06-17': 5853,
  '2026-06-18': 6829,
  '2026-06-19': 8411,
  '2026-06-20': 7236,
  '2026-06-21': 6696,
  '2026-0

**Equivalent:** `next(iter(...))`. Same result, but doesn't build a full list first - matters for memory when queries return many results.

Read it as: "give me the values, then give me the first one."

In [11]:

next(iter(d["query"]["pages"].values()))
# showing how to iterate multiple pages

{'pageid': 15992,
 'ns': 0,
 'title': 'Jimmy Carter',
 'pageviews': {'2026-05-11': 7251,
  '2026-05-12': 7336,
  '2026-05-13': 7281,
  '2026-05-14': 7219,
  '2026-05-15': 6496,
  '2026-05-16': 6009,
  '2026-05-17': 6740,
  '2026-05-18': 6895,
  '2026-05-19': 6708,
  '2026-05-20': 6781,
  '2026-05-21': 6493,
  '2026-05-22': 6087,
  '2026-05-23': 5881,
  '2026-05-24': 6231,
  '2026-05-25': 6465,
  '2026-05-26': 6668,
  '2026-05-27': 6516,
  '2026-05-28': 6673,
  '2026-05-29': 7726,
  '2026-05-30': 6780,
  '2026-05-31': 6481,
  '2026-06-01': 6355,
  '2026-06-02': 6387,
  '2026-06-03': 6414,
  '2026-06-04': 6074,
  '2026-06-05': 6415,
  '2026-06-06': 5758,
  '2026-06-07': 6023,
  '2026-06-08': 5795,
  '2026-06-09': 5695,
  '2026-06-10': 5789,
  '2026-06-11': 5606,
  '2026-06-12': 5746,
  '2026-06-13': 5750,
  '2026-06-14': 6446,
  '2026-06-15': 6455,
  '2026-06-16': 5958,
  '2026-06-17': 5853,
  '2026-06-18': 6829,
  '2026-06-19': 8411,
  '2026-06-20': 7236,
  '2026-06-21': 6696,
  '2026-0

Suppose you had multiple pages to iterate over - we know how to do this! For example, the below will print out the title of every page we return.

In [25]:
for page in d["query"]["pages"].values():
    print(page["title"]) # can be used langlinks count will print langlinks count instead
    #can do fstring

Jimmy Carter


Now we have the page dict. Extracting the count is a normal dictionary lookup:

In [30]:
# Extract the count from the nested response
d = r.json()
page = next(iter(d["query"]["pages"].values()))
count = page["langlinkscount"]

print(f"Jimmy Carter's Wikipedia page is translated into {count} languages.")

Jimmy Carter's Wikipedia page is translated into 164 languages.


## Wikipedia: Jimmy Carter, page views (Your Turn)

In [37]:
parameters = {
    "action":  "query",
    "titles":  "Jimmy_Carter",
    "prop":    "pageviews",
    "format":  "json",
}

r = requests.get(endpoint, params=parameters, headers=HEADERS)
d = r.json()
d

{'batchcomplete': '',
 'query': {'normalized': [{'from': 'Jimmy_Carter', 'to': 'Jimmy Carter'}],
  'pages': {'15992': {'pageid': 15992,
    'ns': 0,
    'title': 'Jimmy Carter',
    'pageviews': {'2026-05-11': 7251,
     '2026-05-12': 7336,
     '2026-05-13': 7281,
     '2026-05-14': 7219,
     '2026-05-15': 6496,
     '2026-05-16': 6009,
     '2026-05-17': 6740,
     '2026-05-18': 6895,
     '2026-05-19': 6708,
     '2026-05-20': 6781,
     '2026-05-21': 6493,
     '2026-05-22': 6087,
     '2026-05-23': 5881,
     '2026-05-24': 6231,
     '2026-05-25': 6465,
     '2026-05-26': 6668,
     '2026-05-27': 6516,
     '2026-05-28': 6673,
     '2026-05-29': 7726,
     '2026-05-30': 6780,
     '2026-05-31': 6481,
     '2026-06-01': 6355,
     '2026-06-02': 6387,
     '2026-06-03': 6414,
     '2026-06-04': 6074,
     '2026-06-05': 6415,
     '2026-06-06': 5758,
     '2026-06-07': 6023,
     '2026-06-08': 5795,
     '2026-06-09': 5695,
     '2026-06-10': 5789,
     '2026-06-11': 5606,
     '202

In [38]:
endpoint = "https://en.wikipedia.org/w/api.php"
parameters = {
    "action":  "query",
    "titles":  "Jimmy_Carter",
    "prop":    "pageviews",
    "format":  "json",
}
r = requests.get(endpoint, params=parameters, headers=HEADERS)
print("Status:", r.status_code)


Status: 200


### Unpacking the pageviews response

Same outer structure as before (`d["query"]["pages"]`), but this time the property we asked for (`pageviews`) is *itself a dictionary*.

In [39]:
d=r.json()
# Same pattern as before
page = next(iter(d["query"]["pages"].values()))
page
#return pagevi

{'pageid': 15992,
 'ns': 0,
 'title': 'Jimmy Carter',
 'pageviews': {'2026-05-11': 7251,
  '2026-05-12': 7336,
  '2026-05-13': 7281,
  '2026-05-14': 7219,
  '2026-05-15': 6496,
  '2026-05-16': 6009,
  '2026-05-17': 6740,
  '2026-05-18': 6895,
  '2026-05-19': 6708,
  '2026-05-20': 6781,
  '2026-05-21': 6493,
  '2026-05-22': 6087,
  '2026-05-23': 5881,
  '2026-05-24': 6231,
  '2026-05-25': 6465,
  '2026-05-26': 6668,
  '2026-05-27': 6516,
  '2026-05-28': 6673,
  '2026-05-29': 7726,
  '2026-05-30': 6780,
  '2026-05-31': 6481,
  '2026-06-01': 6355,
  '2026-06-02': 6387,
  '2026-06-03': 6414,
  '2026-06-04': 6074,
  '2026-06-05': 6415,
  '2026-06-06': 5758,
  '2026-06-07': 6023,
  '2026-06-08': 5795,
  '2026-06-09': 5695,
  '2026-06-10': 5789,
  '2026-06-11': 5606,
  '2026-06-12': 5746,
  '2026-06-13': 5750,
  '2026-06-14': 6446,
  '2026-06-15': 6455,
  '2026-06-16': 5958,
  '2026-06-17': 5853,
  '2026-06-18': 6829,
  '2026-06-19': 8411,
  '2026-06-20': 7236,
  '2026-06-21': 6696,
  '2026-0

Notice: `pageviews` is a whole dict inside the page, not a single number.

- `langlinkscount` was `164` (one number)
- `pageviews` is `{"2026-06-01": 4231, "2026-06-02": 4015, ...}` — one entry per day

The response *shape* depends on the property you asked for. Always inspect before you extract.

In [42]:
# Look at just the pageviews sub-dict
views = page["pageviews"]
views

{'2026-05-11': 7251,
 '2026-05-12': 7336,
 '2026-05-13': 7281,
 '2026-05-14': 7219,
 '2026-05-15': 6496,
 '2026-05-16': 6009,
 '2026-05-17': 6740,
 '2026-05-18': 6895,
 '2026-05-19': 6708,
 '2026-05-20': 6781,
 '2026-05-21': 6493,
 '2026-05-22': 6087,
 '2026-05-23': 5881,
 '2026-05-24': 6231,
 '2026-05-25': 6465,
 '2026-05-26': 6668,
 '2026-05-27': 6516,
 '2026-05-28': 6673,
 '2026-05-29': 7726,
 '2026-05-30': 6780,
 '2026-05-31': 6481,
 '2026-06-01': 6355,
 '2026-06-02': 6387,
 '2026-06-03': 6414,
 '2026-06-04': 6074,
 '2026-06-05': 6415,
 '2026-06-06': 5758,
 '2026-06-07': 6023,
 '2026-06-08': 5795,
 '2026-06-09': 5695,
 '2026-06-10': 5789,
 '2026-06-11': 5606,
 '2026-06-12': 5746,
 '2026-06-13': 5750,
 '2026-06-14': 6446,
 '2026-06-15': 6455,
 '2026-06-16': 5958,
 '2026-06-17': 5853,
 '2026-06-18': 6829,
 '2026-06-19': 8411,
 '2026-06-20': 7236,
 '2026-06-21': 6696,
 '2026-06-22': 6744,
 '2026-06-23': 5746,
 '2026-06-24': 5305,
 '2026-06-25': 5377,
 '2026-06-26': 5461,
 '2026-06-27'

Now `views` is a `{date: count}` dictionary. To use it, iterate with `.items()`:

In [43]:
# Extract the pageviews dict and print each day
page = next(iter(d["query"]["pages"].values()))
views = page["pageviews"]

for date, n in views.items():
    print(f"{date}: {n}")

2026-05-11: 7251
2026-05-12: 7336
2026-05-13: 7281
2026-05-14: 7219
2026-05-15: 6496
2026-05-16: 6009
2026-05-17: 6740
2026-05-18: 6895
2026-05-19: 6708
2026-05-20: 6781
2026-05-21: 6493
2026-05-22: 6087
2026-05-23: 5881
2026-05-24: 6231
2026-05-25: 6465
2026-05-26: 6668
2026-05-27: 6516
2026-05-28: 6673
2026-05-29: 7726
2026-05-30: 6780
2026-05-31: 6481
2026-06-01: 6355
2026-06-02: 6387
2026-06-03: 6414
2026-06-04: 6074
2026-06-05: 6415
2026-06-06: 5758
2026-06-07: 6023
2026-06-08: 5795
2026-06-09: 5695
2026-06-10: 5789
2026-06-11: 5606
2026-06-12: 5746
2026-06-13: 5750
2026-06-14: 6446
2026-06-15: 6455
2026-06-16: 5958
2026-06-17: 5853
2026-06-18: 6829
2026-06-19: 8411
2026-06-20: 7236
2026-06-21: 6696
2026-06-22: 6744
2026-06-23: 5746
2026-06-24: 5305
2026-06-25: 5377
2026-06-26: 5461
2026-06-27: 5861
2026-06-28: 6741
2026-06-29: 6364
2026-06-30: 5460
2026-07-01: 5759
2026-07-02: 5975
2026-07-03: 6577
2026-07-04: 7862
2026-07-05: 7866
2026-07-06: 7579
2026-07-07: 8419
2026-07-08: 76

In [44]:
# Or as a DataFrame
views_df = pd.DataFrame(list(views.items()), columns=["date", "views"])
views_df["date"] = pd.to_datetime(views_df["date"])
views_df.head()

,date,views
0,2026-05-11,7251
1,2026-05-12,7336
2,2026-05-13,7281
3,2026-05-14,7219
4,2026-05-15,6496


## Wikipedia: multiple titles in one request

In [45]:
parameters = {
    "action":  "query",
    "titles":  "Jimmy_Carter|George_H._W._Bush|Bill_Clinton",
    "prop":    "langlinkscount",
    "format":  "json",
}

r = requests.get(endpoint, params=parameters, headers=HEADERS)
d = r.json()
d

{'batchcomplete': '',
 'query': {'normalized': [{'from': 'Jimmy_Carter', 'to': 'Jimmy Carter'},
   {'from': 'George_H._W._Bush', 'to': 'George H. W. Bush'},
   {'from': 'Bill_Clinton', 'to': 'Bill Clinton'}],
  'pages': {'3356': {'pageid': 3356,
    'ns': 0,
    'title': 'Bill Clinton',
    'langlinkscount': 173},
   '11955': {'pageid': 11955,
    'ns': 0,
    'title': 'George H. W. Bush',
    'langlinkscount': 157},
   '15992': {'pageid': 15992,
    'ns': 0,
    'title': 'Jimmy Carter',
    'langlinkscount': 164}}}}

### Unpacking the multi-title response

This time we asked for **three pages** instead of one. The structure is otherwise identical, but `pages` now has three entries.

In [ ]:
d["query"]["pages"]

{'3356': {'pageid': 3356,
  'ns': 0,
  'title': 'Bill Clinton',
  'langlinkscount': 173},
 '11955': {'pageid': 11955,
  'ns': 0,
  'title': 'George H. W. Bush',
  'langlinkscount': 157},
 '15992': {'pageid': 15992,
  'ns': 0,
  'title': 'Jimmy Carter',
  'langlinkscount': 164}}

Three entries under three different page-ID keys.

**We can't use `next(iter(...))` anymore** — that would give us just one of the three. We need to iterate.

In [46]:
## iterate the loop with the three pages.
for page in d["query"]["pages"].values():
    title = page["title"]
    n = page.get("langlinkscount", "n/a") # na is for no information
    print(f"{title}: {n} languages")

Bill Clinton: 173 languages
George H. W. Bush: 157 languages
Jimmy Carter: 164 languages


## Wikipedia: pagination with `continue`

In [48]:
# Category members — too many to fit in a single response
API_URL = "https://en.wikipedia.org/w/api.php"

params = {
    "action":  "query",
    "list":    "categorymembers",
    "cmtitle": "Category:Wikipedians_interested_in_history",
    "cmlimit": "100",
    "format":  "json",
}
#single request for the fisrt
# Just the first batch
resp = requests.get(API_URL, params=params, headers=HEADERS)
resp.raise_for_status()
data = resp.json()

print("Top-level keys:", list(data.keys()))
print(f"Got {len(data['query']['categorymembers'])} items in first batch")
print("Continue token:", data.get("continue"))

# if it is telling continuethat menas not enought data we have got yet,
# for this it is saying 'true', we can do a while/for loop

Top-level keys: ['batchcomplete', 'continue', 'query']
Got 100 items in first batch
Continue token: {'cmcontinue': 'page|2a34343a443a505a04403a3e32405a0a9e2a34343a443a505a04403a3e32405a0430324e2e4c3a48503a543204524e324c042c4658324e01251901dcb7dcb7dcbbdcc2dc08|16906808', 'continue': '-||'}


In [50]:
# Now the full loop — keep fetching until there's no continue token
# CAUTION: for huge categories, you should cap the iteration count.
all_members = []
params = {
    "action":  "query",
    "list":    "categorymembers",
    "cmtitle": "Category:Wikipedians_interested_in_history",
    "cmlimit": "100",
    "format":  "json",
}

MAX_PAGES = 100   # safety cap — increase if you need to fetch more
for _ in range(MAX_PAGES):
    resp = requests.get(API_URL, params=params, headers=HEADERS)
    resp.raise_for_status()
    data = resp.json()

    batch = data["query"]["categorymembers"]
    all_members.extend(batch)
    print(f"Fetched {len(batch)} (total: {len(all_members)})")

    if "continue" not in data:
        break
    params.update(data["continue"])

print(f"\nDone! Total: {len(all_members)}")

Fetched 100 (total: 100)
Fetched 100 (total: 200)
Fetched 100 (total: 300)
Fetched 100 (total: 400)
Fetched 100 (total: 500)
Fetched 100 (total: 600)
Fetched 100 (total: 700)
Fetched 100 (total: 800)
Fetched 100 (total: 900)
Fetched 100 (total: 1000)
Fetched 100 (total: 1100)
Fetched 100 (total: 1200)
Fetched 100 (total: 1300)
Fetched 100 (total: 1400)
Fetched 100 (total: 1500)
Fetched 100 (total: 1600)
Fetched 100 (total: 1700)
Fetched 100 (total: 1800)
Fetched 100 (total: 1900)
Fetched 100 (total: 2000)
Fetched 100 (total: 2100)
Fetched 100 (total: 2200)
Fetched 100 (total: 2300)
Fetched 100 (total: 2400)
Fetched 100 (total: 2500)
Fetched 100 (total: 2600)
Fetched 100 (total: 2700)
Fetched 100 (total: 2800)
Fetched 100 (total: 2900)
Fetched 100 (total: 3000)
Fetched 100 (total: 3100)
Fetched 100 (total: 3200)
Fetched 100 (total: 3300)
Fetched 100 (total: 3400)
Fetched 100 (total: 3500)
Fetched 100 (total: 3600)
Fetched 100 (total: 3700)
Fetched 100 (total: 3800)
Fetched 100 (total: 3

In [62]:
# Convert to DataFrame
members_df = pd.DataFrame(all_members)
members_df.head()

""


## FRED setup

To use FRED you need a free API key.

1. Register at <https://fred.stlouisfed.org/docs/api/api_key.html>
2. In Colab, click the **key icon** in the left sidebar
3. Add a new secret called `FRED_API_KEY` with your key as the value
4. Toggle "Notebook access" on for this notebook
5. Run the cell below — it should print your key (just to confirm the secret is set; you can delete the print line after testing)


#DO NOT PASTE API KEY IN SCRIPT, use secrets in colab

In [77]:
from google.colab import userdata
#for positron, API_KEY= userdata.get(paste API Key)
API_KEY = "abcdffghijklmnopqrstuvwxyz123456"
print("Key length:", len(API_KEY))   # should print a number if the secret is set

Key length: 32


### Finding FRED parameters

FRED doesn't have an interactive sandbox like Wikipedia, but the docs are usable once you know where to look.

**Official docs:** <https://fred.stlouisfed.org/docs/api/fred/>

Every endpoint has its own page listing every accepted parameter, plus a **clickable example URL** you can hit in your browser to see the JSON response.

**Finding `series_id`** (the most important parameter):

The docs don't list series IDs (there are a ton). Instead:

1. Go to <https://fred.stlouisfed.org/>
2. Search for what you want ("unemployment", "GDP", "inflation")
3. Click into a series page — **the URL ends with the `series_id`**
4. Example: `fred.stlouisfed.org/series/UNRATE` → `series_id = "UNRATE"`

**Endpoints worth knowing:**

- `/series/observations` — time series data (what we've been using)
- `/series/search` — search for series by keyword
- `/category/children` — browse the category tree
- `/release/series` — series from a particular data release

**Common parameters for `/series/observations`:**

| Parameter | Meaning |
|---|---|
| `series_id` | Required. The series identifier from FRED's site. |
| `api_key` | Required. Your key. |
| `file_type=json` | Always set this. Otherwise you get XML. |
| `observation_start` | Filter by start date (`YYYY-MM-DD`). |
| `observation_end` | Filter by end date. |
| `units` | Pre-compute changes: `chg`, `pch`, `pca` (percent change from a year ago). |
| `frequency` | Resample: `m` (monthly), `q` (quarterly), `a` (annual). |
| `limit` | Cap the number of observations returned. |


## FRED: US unemployment rate

In [80]:
endpoint = "https://api.stlouisfed.org/fred/series/observations"
#every website last words
API_KEY = "abcdefghijklmnopqrstuvwxyz123456"

params = {
    "series_id": "UNRATE",
    "api_key":   API_KEY,
    "file_type": "json",
}

r = requests.get(endpoint, params=params)
print("Status:", r.status_code)
data = r.json()
print("Number of observations:", len(data["observations"]))


Status: 400


KeyError: 'observations'

### Unpacking the FRED response

FRED returns a different item than Wikipedia. Let's inspect before we convert.

In [73]:
data.keys()

dict_keys(['error_code', 'error_message'])

Top-level metadata (`realtime_start`, `count`, `units`, etc.) plus one key we care about: **`observations`** , which is a **list** of records.

Different from Wikipedia's dict-of-dicts. FRED gives you a list where each entry is one time-point.

In [72]:
# Peek at the first few observations
data["observations"][:3]

KeyError: 'observations'

Each observation is a dict with the same keys: `date`, `value`, and some bookkeeping stuff. Both `date` and `value` come back as **strings** (`"1948-01-01"`, `"3.4"`).

Two consequences:

- Building a DataFrame from this list is easy
- But we'll have to convert `date` to datetime and `value` to numeric before doing math or plotting

In [ ]:
# Convert to a DataFrame with proper types
obs = pd.DataFrame(data["observations"])
obs["date"]  = pd.to_datetime(obs["date"])
obs["value"] = pd.to_numeric(obs["value"], errors="coerce")

obs = obs[["date", "value"]]
obs.tail(10)

In [ ]:
# Filter to a date range
recent = obs[obs["date"] >= "2020-01-01"]
recent.head()

In [ ]:
# Quick plot
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(recent["date"], recent["value"])
plt.xlabel("Date")
plt.ylabel("Unemployment rate (%)")
plt.title("US unemployment rate, 2020-")
plt.tight_layout()
plt.show()

## FRED: try a different series

The series_id parameter is the key. Browse the FRED catalog at <https://fred.stlouisfed.org/> for ideas. A few to try:

- `GDP` — Gross Domestic Product
- `CPIAUCSL` — Consumer Price Index (inflation)
- `DGS10` — 10-year Treasury yield
- `MEDLISPRIPERSQUFEE` — Median listing price per square foot